In [3]:
# Clone repos + install
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy activation scales from Drive
!mkdir -p smoothquant_repo/act_scales
!cp /content/drive/MyDrive/thesis_results/act_scales/*.pt smoothquant_repo/act_scales/

# Create Drive output folder
!mkdir -p /content/drive/MyDrive/thesis_results/task01_alphasweep

# Verify
!nvidia-smi
!ls -la smoothquant_repo/act_scales/
!python -c "from smoothquant.smooth import smooth_lm; print('smoothquant OK')"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 53 (delta 13), reused 50 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 29.18 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/llm-quantization-thesis
fatal: destination path 'smoothquant_repo' already exists and is not an empty directory.
Found existing installation: smoothquant 0.0.0
Uninstalling smoothquant-0.0.0:
  Successfully uninstalled smoothquant-0.0.0
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Apr 15 00:22:27 2026       
+-----------------------------------------------------------------------------------------+

In [4]:
import sys
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_model
from datasets import load_dataset
import json, os, time, copy, tqdm

MODEL = "facebook/opt-1.3b"
SCALES = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/opt-1.3b.pt"
ALPHAS = [0.1, 0.3, 0.5, 0.7, 0.9]  # step=0.2, 2 below and 2 above paper's 0.5
SAVE_DIR = "/content/drive/MyDrive/thesis_results/task01_alphasweep"

CONFIGS = {
    "SQ-O1":     {"weight_quant": "per_tensor",  "act_quant": "per_token"},
    "SQ-PCW-PT": {"weight_quant": "per_channel", "act_quant": "per_token"},
}


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]), return_tensors="pt"
        ).input_ids.to(device)
        self.n_samples = n_samples

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n = self.n_samples if self.n_samples else self.dataset.size(1) // 2048
        for i in tqdm.tqdm(range(n), desc="Evaluating"):
            batch = self.dataset[:, (i * 2048):((i + 1) * 2048)].to(model.device)
            lm_logits = model(batch).logits
            shift_logits = lm_logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048):((i + 1) * 2048)][:, 1:]
            loss = nn.CrossEntropyLoss()(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
            )
            nlls.append(loss.float() * 2048)
        return torch.exp(torch.stack(nlls).sum() / (n * 2048))


# --- Load once ---
print("Loading tokenizer + dataset...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
evaluator = Evaluator(dataset, tokenizer, "cuda")
act_scales = torch.load(SCALES)

print("Loading base model weights (will reload per run)...")
# We just verify it loads; actual loading happens in the loop
_ = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="cpu")
del _
torch.cuda.empty_cache()

all_results = []

total_runs = len(ALPHAS) * len(CONFIGS)
run_num = 0

for alpha in ALPHAS:
    for config_label, qparams in CONFIGS.items():
        run_num += 1
        print(f"\n{'='*60}")
        print(f"  Run {run_num}/{total_runs}: {config_label} alpha={alpha}")
        print(f"  Weight: {qparams['weight_quant']}, Act: {qparams['act_quant']}")
        print(f"{'='*60}")

        start = time.time()

        # Fresh model load each run (smoothing modifies weights in-place)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.bfloat16, device_map="auto"
        )

        # Smooth
        smooth_lm(model, act_scales, alpha)

        # Quantize
        model = quantize_model(
            model,
            weight_quant=qparams["weight_quant"],
            act_quant=qparams["act_quant"],
            quantize_bmm_input=True,
        )

        # Evaluate
        ppl = evaluator.evaluate(model)
        elapsed = time.time() - start
        ppl_val = ppl.item()

        result = {
            "config_label": config_label,
            "model": MODEL,
            "alpha": alpha,
            "weight_quant": qparams["weight_quant"],
            "act_quant": qparams["act_quant"],
            "wikitext2_ppl": round(ppl_val, 4),
            "duration_seconds": round(elapsed, 1),
        }
        all_results.append(result)

        # Save individual result
        fname = f"opt-1.3b_{config_label}_a{alpha}.json"
        with open(os.path.join(SAVE_DIR, fname), "w") as f:
            json.dump(result, f, indent=2)

        print(f">>> {config_label} alpha={alpha}: PPL = {ppl_val:.4f} ({elapsed:.0f}s)")

        # Cleanup
        del model
        torch.cuda.empty_cache()

# --- Save combined results ---
with open(os.path.join(SAVE_DIR, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)

# --- Print summary table ---
print(f"\n\n{'='*60}")
print(f"  ALPHA SWEEP RESULTS — OPT-1.3B")
print(f"{'='*60}")
print(f"\n{'Alpha':>6} {'SQ-O1 PPL':>12} {'SQ-PCW-PT PPL':>15} {'Diff (O1-C)':>12} {'C wins?':>8}")
print("-" * 55)

for alpha in ALPHAS:
    o1 = next((r for r in all_results if r["alpha"] == alpha and r["config_label"] == "SQ-O1"), None)
    c  = next((r for r in all_results if r["alpha"] == alpha and r["config_label"] == "SQ-PCW-PT"), None)
    if o1 and c:
        diff = o1["wikitext2_ppl"] - c["wikitext2_ppl"]
        wins = "YES" if diff > 0 else "no"
        print(f"{alpha:>6.1f} {o1['wikitext2_ppl']:>12.4f} {c['wikitext2_ppl']:>15.4f} {diff:>+12.4f} {wins:>8}")

print(f"\nPositive diff = C is better (lower PPL). Results saved to {SAVE_DIR}")

Loading tokenizer + dataset...
Loading base model weights (will reload per run)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



  Run 1/10: SQ-O1 alpha=0.1
  Weight: per_tensor, Act: per_token


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Evaluating: 100%|██████████| 40/40 [02:06<00:00,  3.17s/it]


>>> SQ-O1 alpha=0.9: PPL = 14.7540 (185s)

  Run 10/10: SQ-PCW-PT alpha=0.9
  Weight: per_channel, Act: per_token


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Evaluating: 100%|██████████| 40/40 [02:07<00:00,  3.18s/it]


>>> SQ-PCW-PT alpha=0.9: PPL = 14.6171 (186s)


  ALPHA SWEEP RESULTS — OPT-1.3B

 Alpha    SQ-O1 PPL   SQ-PCW-PT PPL  Diff (O1-C)  C wins?
-------------------------------------------------------
   0.1      16.0183         15.6460      +0.3723      YES
   0.3      14.9330         14.8997      +0.0333      YES
   0.5      14.6864         14.7017      -0.0153       no
   0.7      14.7775         14.6422      +0.1353      YES
   0.9      14.7540         14.6171      +0.1369      YES

Positive diff = C is better (lower PPL). Results saved to /content/drive/MyDrive/thesis_results/task01_alphasweep
